### Installations 

In [1]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd 
#pd.options.display.float_format = '{:.2%}'.format

import numpy as np
import os
from os import path 

import investpy
import ray 
from timebudget import timebudget

from datetime import date, timedelta
import datetime 

from yahoofinancials import YahooFinancials
import yfinance as yf
import quantstats as qs

import pyfolio as pf

from pypfopt import expected_returns
from pypfopt import plotting
import ffn

import rpy2
import riskfolio as rp

In [11]:
!pip uninstall rpy2

zsh:1: /usr/local/bin/pip: bad interpreter: /System/Library/Frameworks/Python.framework/Versions/2.7/Resources/Python.app/Contents/MacOS/Python: no such file or directory


In [9]:
%load_ext rpy2.ipython

OSError: cannot load library 'D:\Install\R\R-3.6.1/lib/libR.dylib': dlopen(D:\Install\R\R-3.6.1/lib/libR.dylib, 0x0002): tried: 'D:\Install\R\R-3.6.1/lib/libR.dylib' (no such file), '/System/Volumes/Preboot/Cryptexes/OSD:\Install\R\R-3.6.1/lib/libR.dylib' (no such file), '/Users/safishajjouz/opt/anaconda3/envs/myport_management_env/lib/D:\Install\R\R-3.6.1/lib/libR.dylib' (no such file), '/System/Volumes/Preboot/Cryptexes/OS/Users/safishajjouz/opt/anaconda3/envs/myport_management_env/lib/D:\Install\R\R-3.6.1/lib/libR.dylib' (no such file), '/Users/safishajjouz/opt/anaconda3/envs/myport_management_env/lib/D:\Install\R\R-3.6.1/lib/libR.dylib' (no such file), '/System/Volumes/Preboot/Cryptexes/OS/Users/safishajjouz/opt/anaconda3/envs/myport_management_env/lib/D:\Install\R\R-3.6.1/lib/libR.dylib' (no such file), '/Users/safishajjouz/opt/anaconda3/envs/myport_management_env/lib/python3.8/site-packages/../../D:\Install\R\R-3.6.1/lib/libR.dylib' (no such file), '/Users/safishajjouz/opt/anacon

In [3]:
%%R 

# load R packages 
install.packages("pacman", repos='http://cran.us.r-project.org', quiet=TRUE)

pacman::p_load(BatchGetSymbols,TTR,tidyquant, rvest,
               plotly,paletteer, ggplot2, 
               PerformanceAnalytics,
               lubridate,stringr,janitor,
               tidyverse,timetk,scales,
               quantmod,readr)


UsageError: Cell magic `%%R` not found.


In [ ]:
# set working directery 
os.chdir("/Users/safishajjouz/GitHub/myPortfolioManagement/files")

In [ ]:
# load my ISA portfolio 
myfunds = pd.read_csv("Performance_isa.csv", skiprows=  3 )

# compute Portfolio weights 
myfunds = myfunds.groupby(['ISIN', "Fidelity Fund Name"]).sum(['Units','Value (£)', 'Book Cost (£)', 'Total Income Paid Out (£)']).reset_index()
myfunds = myfunds.groupby(['ISIN', "Fidelity Fund Name"]).sum(['Units','Value (£)', 'Book Cost (£)', 'Total Income Paid Out (£)']).reset_index()

# I chose my weights based on how much I paid
myfunds["weight"] = round((myfunds["Book Cost (£)"]/pd.Series(myfunds['Book Cost (£)']).sum()),4)

# clean the dataframe 
myfunds = myfunds.rename(columns = {"Fidelity Fund Name": 'fund', 
                                    'Total Return Since Inception (£)':'Gain/Loss',
                                   'Gain/loss Since Inception (%)':'return'}, inplace = False)
# keep columns 
myfunds = myfunds[['fund', 'ISIN', "weight", 'Gain/Loss', 
                   'Book Cost (£)', 'Value (£)', "return"]]

# clean strings 
myfunds.columns = [s.replace ('(£)',"") for s in myfunds.columns]
myfunds.columns = [s.replace ('(p)',"") for s in myfunds.columns]

# sort per asset holdings 
myfunds = myfunds.sort_values('weight', ascending = False)
myfunds['Date_of_access'] = date.today()
myfunds

In [ ]:
df_date_aquired = pd.read_csv("funds_date_aquired.csv")

In [ ]:
df_date_aquired['Date Acquired'] = pd.to_datetime(df_date_aquired['Date Acquired'],format='%d/%m/%Y')
df_date_aquired['days_in_posession'] = pd.to_datetime(date.today().strftime("%d/%m/%Y"), format = '%d/%m/%Y') - pd.to_datetime(df_date_aquired['Date Acquired'])


df_date_aquired['Date_opened_ISA'] = pd.to_datetime('23/10/2020',format='%d/%m/%Y')
df_date_aquired['days_ISA'] = pd.to_datetime(date.today().strftime("%d/%m/%Y"), format = '%d/%m/%Y')- pd.to_datetime(df_date_aquired['Date_opened_ISA'])

myfunds = myfunds.merge(df_date_aquired)
myfunds = myfunds.drop('Fidelity Fund Name', axis =1)

In [ ]:
# short names 
#funds_short_names = pd.read_excel("list_of_funds.xlsx", engine='openpyxl')  # to be used later
#funds_short_names = funds_short_names[["short_name", "ISIN", 'yahoo_ticker']]
#funds_short_names

In [ ]:
#myfunds = pd.merge(myfunds,funds_short_names)
#myfunds["fund"] = myfunds["short_name"]

# convert time-delta series to numeric 
myfunds['days_in_posession']  = myfunds.days_in_posession.dt.days

In [ ]:
myfunds = myfunds[['fund','ISIN', 'weight','return', 'days_in_posession']]
myfunds

## Load Fund Prices 

In [ ]:
# dates for the prices  
from_date = '01/01/2000'    
to_date   =  date.today() -  timedelta(days=1)
to_date   = to_date.strftime("%d/%m/%Y") 

In [ ]:
mypath    = '/Users/safishajjouz/GitHub/myPortfolioManagement/files'
file_name = "my_main_database.pkl"
file_path_to_load = path.join(mypath, file_name)

# load prices 
df = pd.read_pickle(file_path_to_load)
df.head()


In [ ]:
## my portfolio 
df_my_portfolio = df[['stock', 'isin']].drop_duplicates().merge(myfunds, on = 'yahoo_ticker')
#my_stocks = df_my_portfolio['stock'].to_list() + ['SCOTTISH MORTGAGE INVESTMENT TR']
my_stocks = df_my_portfolio['stock'].to_list() 
my_stocks

In [ ]:
df_R = df.reset_index()
df_R = df_R[['Date', 'adjclose', 'stock', 'yahoo_ticker']]
df_R

In [ ]:
'''
%%R -i tickers -i from_date -i to_date -i df


first.date <- as.Date(from_date, format = "%m/%d/%Y")
last.date <- as.Date(to_date, format = "%m/%d/%Y")

# download prices 
future::plan(future::multisession, workers = floor(parallel::detectCores()-1))
mystocks <- BatchGetSymbols::BatchGetSymbols(tickers =tickers,
                                          first.date = first.date,
                                          last.date = last.date, 
                                          freq.data = 'daily',
                                          be.quiet = TRUE,
                                          thresh.bad.data = 0.10, 
                                          do.complete.data = T,
                                          do.fill.missing.prices = F,
                                          do.parallel = T,
                                          do.cache = F
)

df_stocks = mystocks$df.tickers %>% filter(volume > 0)

df_stocks = df_stocks %>% dplyr::rename(Date = ref.date, Close = price.adjusted, fund = ticker)%>%
           subset(select = c(Date, Close, fund))
df_stocks$ISIN = df_stocks$fund

df_R = rbind(df_stocks, df)
df_R$Date = as.Date(df_R$Date)
'''

In [ ]:
%%R -i df_R
df_R = df_R # assign the df_R
df_R$Date = as.Date(df_R$Date)
df_R %>% head()

#### Calculate Returns

There are various ways to calculate returns. For portfolio construction you might want to calculate returns based on log differences. But for performance you might want simple arithmetic. Hence, chose carefully. 



In [ ]:
%%R

# method to calculate returns over an interval 
return_method = c('log', 'arithmetic')[2]

# over which period (interval) to calculate returns 
period_returns = c('daily', 'weekly', 'monthly', 'quarterly', 'yearly')[1]

# number of periods in a year 
scale = c(NA, 252, 12, 4)[1]

# colname of returns 
returns_col_name = 'Ra'

# my_assets_col_name
my_assets_col_name = 'stock'

price_colname = 'adjclose'

# calculation of returns and keep the dataframe in long format 
df_returns = df_R %>% group_by(stock) %>%
                tq_transmute(select = adjclose,
                             mutate_fun = periodReturn,
                             period= period_returns,
                             type = return_method,
                             col_rename = returns_col_name)
df_returns %>% head()

In [ ]:
%%R -i df_my_portfolio 

# calculate portfolio returns 
wts_map <- tibble(symbols = df_my_portfolio$stock,
                      weights = df_my_portfolio$weight)

# my portfolio returns 
df_portfolio_ISA = df_returns %>% filter(stock %in% df_my_portfolio$stock)  %>% 
                   tq_portfolio(assets_col  = stock, returns_col = Ra, weights = wts_map,
                                col_rename  = "Ra", 
                                geometric = FALSE, 
                                contribution = FALSE,
                                wealth.index = FALSE) 

df_portfolio_ISA$stock='portfolio_ISA'

# merge with fund prices 
df_returns = rbind(df_returns, df_portfolio_ISA)

# wide format 
returns_xts = df_returns %>% spread( key = stock, value = Ra) %>% timetk::tk_xts(date_var = Date, silent = T)


#### Contributions Returns 

In [ ]:
%%R 


names(returns_xts)[which(names(returns_xts)=='Vanguard FTSE Developed World e')]<-'world_ex_uk'

In [ ]:
%%R -i df_my_portfolio -w 15 -h 10 --units in -r 80

date_to_check = Sys.Date() - 365


benmark_comp_returns = returns_xts %>% window(start = date_to_check) %>% apply.monthly(Return.cumulative) %>%             # calculate compound returns per month 
                       fortify()%>%                 
                       mutate(Date = lubridate::floor_date(Index, "month")) %>%
                       janitor::clean_names() %>% 
                       subset(select = c(date, world_ex_uk, portfolio_isa)) %>%
                       dplyr::rename(Date = date, Benchmark_returns = world_ex_uk) %>%
                       reshape2::melt(id.vars="Date",                   # reshape the Df to long-format 
                                 variable.name="stock",
                                 value.name="Ra_contributions")


                    
df_fund_contributions = df_returns %>% filter(stock %in% df_my_portfolio$stock)  %>%  #filter 
        filter(Date >= date_to_check) %>%            # slice at this date 
        tq_portfolio(                               # compute contributions 
        assets_col  = stock,
        returns_col = Ra,
        weights     = wts_map,
        col_rename  = "portfolio_ISA", 
                   contribution = TRUE,
                   wealth.index = FALSE) %>% 
        subset(select = - c(Residual, portfolio.returns))%>%                # drop columns 
        timetk::tk_xts(date_var = Date, silent = T) %>%  # convert to xts object 
        apply.monthly(Return.cumulative) %>%             # calculate compound returns per month 
        fortify()%>%                                     # convert to dataframe object 
        mutate(Date = lubridate::floor_date(Index, "month")) %>% # floor date to start 1st of the month 
        subset(select = -Index) %>%
        reshape2::melt(id.vars="Date",                   # reshape the Df to long-format 
                       variable.name="stock",
                        value.name="Ra_contributions") 

myplot_cont <- ggplot2::ggplot(df_fund_contributions, aes(x = Date, y = Ra_contributions*100)) + 
               geom_col(aes(fill=stock)) +
               geom_point(data = benmark_comp_returns, aes(color=stock)) + 
               scale_y_continuous(n.breaks = 15)+ 
               scale_color_manual(labels = c("Benchmark_Returns", 'portfolio_isa'), 
                                  values = c("black", 'red')) + 
               ylab('Returns Contributions (%)')+
               scale_fill_brewer(palette = "Paired") +
               scale_x_date(breaks = scales::breaks_pretty(15)) +
               theme_minimal()+theme(legend.position="bottom", 
                                     text = element_text(size=20), 
                                     legend.title = element_blank(),
                                    axis.text.x = element_text(angle = 45, vjust = 1, hjust = 1))+
               guides(color = guide_legend(nrow = 2))
p <-ggplotly(myplot_cont)
htmlwidgets::saveWidget(p, "monthly_contributions.html")
p

In [ ]:
%%R -w 18 -h 8 --units in -r 80

df_new<-rbind(df_fund_contributions) %>%
  mutate(contrib = 100*Ra_contributions)

ggplot(df_new) +  
  geom_col(aes(x = stock, y = contrib, fill = contrib >0)) +
  #geom_point(data = benmark_comp_returns, aes(x = Fund, y = Ra_contributions*100, color=Fund)) +
  scale_fill_brewer(palette = "Set1")+
  geom_hline(yintercept = 0) +
  facet_grid(~Date) +
  coord_flip() + theme(text = element_text(size=20), 
                       axis.title.y =element_blank(),
                       axis.title.x =element_blank(),
                       axis.text.x = element_text(angle = 45, vjust = 1, hjust = 1),
                       axis.text.y = element_text(angle = 45, vjust = 1, hjust = 1),
                       legend.position = "none")

In [ ]:
%%R -w 18 -h 8 --units in -r 80

benmark_comp_returns %>%
mutate(contrib = 100*Ra_contributions) %>%
ggplot() +  
  geom_col(aes(x = stock, y = contrib, fill = stock)) +
  #geom_point(data = benmark_comp_returns, aes(x = Fund, y = Ra_contributions*100, color=Fund)) +
  scale_fill_brewer(palette = "Set1")+
  geom_hline(yintercept = 0) +
  facet_grid(~Date)+theme(text = element_text(size=20), 
                       axis.title.y =element_blank(),
                       axis.title.x =element_blank(),
                       axis.text.x =element_blank(),
                       axis.text.y =element_text(angle = 45, vjust = 1, hjust = 1),
                         legend.position = "bottom", legend.title = element_blank())->plot2
p2 <-ggplotly(plot2)
htmlwidgets::saveWidget(p2, "monthly_per_rel_to_benchmark.html")
plot2

In [ ]:
%matplotlib inline


# extend pandas functionality with metrics, etc.
qs.extend_pandas()

# fetch the daily returns for a stock
stock = qs.utils.download_returns('FB')

# show sharpe ratio
qs.stats.sharpe(stock)

# or using extend_pandas() :)
stock.sharpe()

In [ ]:
%%R
returns = fortify(returns_xts) %>% dplyr::rename(Date=Index)

In [ ]:
returns = %Rget returns
returns['Date'] = pd.to_datetime(returns['Date'], unit='D')

In [ ]:
returns = returns.set_index('Date')

In [ ]:
qs.reports.full(returns['portfolio_ISA'], "SPY")

A few quick observations:
 
The strategy looks that is over-fitted since my management skills reacted after the events. Before April 2021, UK and Baillie Gifford funds had a good momentum which gave some alpha. Later, when the momentum lost. Index looks that has beaten me. 



#### Compound (cumulative) Returns 

In [ ]:
%%R

# compound returns for each asset (using geometric method)
compound_geom = returns_xts %>% Return.cumulative() %>% t()
colnames(compound_geom)<-gsub(" ", "",colnames(compound_geom))
colnames(compound_geom)<-'CumReturns_per_fund_sample'

# compound returns for each asset (using arithmetic method)
compound_non_geom = returns_xts %>% window(start = '2020-11-01') %>% 
                     Return.cumulative(geometric=FALSE) %>% t() 
                  
colnames(compound_non_geom)<-gsub(" ", "",colnames(compound_non_geom))
colnames(compound_non_geom)<-'CumReturn_since_ISA'

# dataframe with compound returns 
df_cum_returns = cbind(compound_non_geom, compound_geom)
df_cum_returns

### Comparing Investments 

- CAGR: $(\frac{Final Value}{First Value})^{1/years} -1$ The formula washes out volatility and shows what's the average or steady state of rate of return.

In [ ]:
%%R -i df_overview

df_cagr = full_join(df_overview, df_R %>% group_by(fund) %>% mutate(
        Last_price = last(Close),
        First_price = first(Close),
    
      ) %>% subset(select = c(fund, Last_price, First_price)) %>% unique(), by = 'fund')

df_cagr$CAGR = ((df_cagr$Last_price)/(df_cagr$First_price))^(1/df_cagr$years_available) - 1

df_cum_returns = full_join(as.data.frame(df_cum_returns) %>% rownames_to_column(var = "fund"), df_cagr, by='fund')
df_returns_overview = df_cum_returns %>% 
                      subset(select = c(fund,CAGR, CumReturns_per_fund_sample, years_available)) %>%
                      filter(fund!='portfolio_ISA')

In [ ]:
%%R
# How to average returns 
# Geometric average compounds the effect of the volatility , simple average does not 
geometric_returns = T

# Minimum Acceptable returns in the same periodicity as returns
MAR = mean(returns$`World ex-UK`, na.rm = T)


#df1 =  returns %>% table.AnnualizedReturns(scale = scale, geometric = geometric_returns) %>% t()
#df2 =  returns %>% table.DownsideRiskRatio(MAR = MAR, scale = scale, digits = 4) %>% t() 
 

# Annualised returns Stats 
df_returns_perf = df_returns %>% group_by(stock) %>%
                     subset(select = c(stock, Date, Ra)) %>%
                     tq_performance(Ra = Ra,
                                    performance_fun = table.AnnualizedReturns, 
                                    scale = scale, geometric = T)
# Adjusted Sharpe 
df_returns_perf = full_join(df_returns_perf, df_returns %>% group_by(stock) %>%
                                             subset(select = c(stock, Date, Ra)) %>%
                                             tq_performance(Ra = Ra,
                                             performance_fun =  AdjustedSharpeRatio, 
                                             scale = scale, geometric = geometric_returns) %>%
                                             dplyr::rename(AdjustedSharpeRatio = `AnnualizedSharpeRatio(Rf=0%)`) )

# Downside Risk Stats 
df_risk_perf = df_returns %>% group_by(stock) %>%
                 subset(select = c(stock, Date, Ra)) %>%
                 tq_performance(Ra = Ra,
                   performance_fun = table.DownsideRiskRatio, 
                   MAR = MAR, scale = scale, digits = 4) %>%
                   subset(select = c(stock, Sortinoratio, Upsidepotentialratio))

df_performance = full_join(df_risk_perf,df_returns_perf)
#df_performance = full_join(df_performance, df_returns_overview, by = 'stock')

In [ ]:
df_performance =%Rget df_performance
dtale.show(df_performance.round(2))

### Trailing Returns 

In [ ]:
%%R 
df_trailling_returns =  table.TrailingPeriods(as.data.frame(returns), 
                        periods=c(30,90, 120, 240), digits = 2, 
                        FUNCS = "Return.cumulative", geometric=TRUE) %>% 
                        t() %>% as.data.frame() %>% 
                        rownames_to_column(var = "stock") %>% 
                        janitor::clean_names()

In [ ]:
df_trailling_returns = %Rget df_trailling_returns
dtale.show(df_trailling_returns)

In [ ]:
%%R
sharpe = SharpeRatio(returns, Rf = 0, p = 0.95, FUN = c("StdDev", "VaR", "ES")[1], annualize = F) 
Ref_sharpe = rep(sharpe[, c('World ex-UK')], ncol(sharpe))
sharpe_temp = sharpe
sharpe_temp[1, ] = Ref_sharpe

# probabilistic sharp (sharpe must be non-annumalized)
prob_sharpe = ProbSharpeRatio(R = returns, Rf = 0,refSR = sharpe_temp, p = 0.95)
prob_sharpe

In [ ]:
%%R 

sharpe = SharpeRatio(returns, Rf = 0, p = 0.95, FUN = c("StdDev", "VaR", "ES")[1], annualize = F) 
Ref_sharpe = rep(sharpe[, c('World ex-UK')], ncol(sharpe))
sharpe_temp = sharpe
sharpe_temp[1, ] = Ref_sharpe
sharpe_temp

In [ ]:
%%R 
returns = fortify(returns_xts) %>% dplyr::rename(Date=Index)
#returns_zero_fill = zerofill(returns) %>% dplyr::rename(Date=Index)
#returns_zero_fill$Date = as.Date(returns_zero_fill$Date, format = '%y-%m-%d')
#returns_zero_fill$Date

In [ ]:
returns = %Rget returns_xts
returns['Date'] = pd.to_datetime(returns['Date'], unit='D')
returns = returns.set_index('Date')
returns[['portfolio_ISA', 'World ex-UK']][returns.index>='2021-04-01'].dropna()

In [ ]:
# plot cumulative returns 
pf.timeseries.cum_returns(returns[['portfolio_ISA', 'World ex-UK']][returns.index>='2021-04-01'].dropna(),
                          starting_value=1).plot(figsize=(12,12))

In [ ]:
price_index = ffn.core.to_price_index(returns, start=1)

df_stats = ffn.core.GroupStats(price_index).stats.dropna()
df_stats = df_stats.transpose()
df_stats = df_stats.drop(['start', 'end', 'rf', 'ytd', 'yearly_sortino', 'yearly_mean',
                          'yearly_skew', 'yearly_kurt', 'twelve_month_win_perc',
                         'worst_year', 'best_year'], axis = 1)

qgrid.show_grid(df_stats)

In [ ]:
price_index = ffn.core.to_price_index(returns.set_index('Date'), start=1)
df_stats = ffn.core.GroupStats(price_index).stats.dropna()
df_stats = df_stats.transpose()
df_stats = df_stats.drop(['start', 'end', 'rf', 'ytd', 'yearly_sortino', 'yearly_mean',
                          'yearly_skew', 'yearly_kurt', 'twelve_month_win_perc',
                         'worst_year', 'best_year'], axis = 1)

qgrid.show_grid(df_stats)

In [ ]:
price_index.to_csv('price_index.csv')

### (Downside) Risk Metrics 


-  <span style="color:blue">**Semi-Deviation**</span>      : Variability of underperformance below mean or zero returns.  
-  <span style="color:blue">**Downside Deviation**</span> : Variability of underperformance below a target. Compares downside risk between assets 
- <span style="color:blue">**Sortino Ratio**</span>: $ \frac{mean(R-MAP)}{\text{Downside Deviation}}$
- <span style="color:blue">**Upside Potential Ratio**</span>
- <span style="color:blue">**Adjusted Sharpe Ratio**</span> : Adjusts the (Annualized) Sharpe ratio for any excess Skweness and Kurtossis of returns.
-  Gain / Loss Deviation     : 
-  Historical (modified) VaR : 
-  Historical (modified) ES  :
 

In [ ]:
%%R 

#returns %>% table.DownsideRisk(MAR = 0, scale = scale, digits = 4) %>% t() 
                   #%>% subset(select = c(`Sortino ratio`))

method_downside_dev = c("full", "subset")[1] # Full or subset of the series below the MAR as the denominator

MAR = mean(returns$`World ex-UK`, na.rm = T)

rbind(DownsideDeviation(returns, MAR = MAR, method = method_downside_dev, potential = FALSE),
      DownsideDeviation(returns, MAR = MAR, method = method_downside_dev, potential = TRUE),
      DownsidePotential(returns, MAR = MAR),
      SemiDeviation(returns, SE = FALSE, SE.control = NULL),
      SemiVariance(returns))

### My Portfolio Analytics 

In [ ]:
# Clean database 
df_cleaned = df.pivot_table(index=["Date"], 
                    columns='fund', 
                    values='Close')
df_cleaned = df_cleaned[df_cleaned.index>='2018-05-01']
df_cleaned = df_cleaned.interpolate(method = "time")
df_cleaned = df_cleaned.rolling(60).mean()

In [ ]:
# calculate returns
df_returns = expected_returns.returns_from_prices(df_cleaned.interpolate(method = "time"))
df_returns = df_returns.interpolate(method = 'time', limit_direction='Both')

### Assessing Pefomance 

In [ ]:
# plot cumulative returns 
date_to_cut = '2020-10-23'
pf.timeseries.cum_returns(df_returns[df_returns.index>=date_to_cut], starting_value=1).plot(figsize=(12,12))

In [ ]:
# check some basic perfomance stats for each asset 
df_stats = ffn.core.GroupStats(df_cleaned[df_cleaned.index>=date_to_cut]).stats
keep_index = ["total_return", "max_drawdown", "calmar", "ytd", 
              "three_month", "six_month", 'one_year', 'three_year',
              'monthly_sharpe']
df_stats[df_stats.index.isin(keep_index)].transpose().sort_values(by=['three_month'], ascending= False)

###  Clusterring of Assets 

In [ ]:
dct = df_returns.calc_ftca(threshold=0.8) # optimal threshold 

df_cluster = pd.DataFrame.from_dict(dct, orient='index')
df_cluster = (pd.DataFrame.from_dict(dct, orient='index').T
   .melt(var_name='cluster', value_name='fund')
   .dropna(subset=['fund']))
df_cluster['cluster'] = [f"cluster_{label}" for label in df_cluster['cluster']]
df_cluster

### Portfolio optimisation 

In [ ]:
import riskfolio as rp

ret_under_training = df_returns[df_returns.index<='2021-03-01']

# Building the portfolio object
port = rp.HCPortfolio(returns=ret_under_training)

# Estimate optimal portfolio:

models = ['HRP', 'HERC', 'HERC2'] # Could be HRP or HERC
codependence = ['mutual_info', 'tail'][0] # Correlation matrix used to group assets in clusters
rf = 0.01 # Risk free rate
linkage = 'ward' # Linkage method used to build clusters
max_k = 10 # Max number of clusters used in two difference gap statistic
leaf_order = True # Consider optimal order of leafs in dendrogram

# risk objectives 
rms = ['vol', 'MV', 'MAD', 'MSV', 'FLPM', 'SLPM',
       'VaR','CVaR', 'EVaR', 'WR', 'MDD', 'ADD',
       'DaR', 'CDaR', 'EDaR', 'UCI', 'MDD_Rel', 'ADD_Rel',
       'DaR_Rel', 'CDaR_Rel', 'EDaR_Rel', 'UCI_Rel']

w_s = pd.DataFrame([])
for model in models:
    for i in rms:
        w = port.optimization(model=model,
                          codependence=codependence,
                          rm=i,
                          rf=rf,
                          covariance='ledoit',
                          linkage=linkage,
                          max_k=max_k,
                          leaf_order=leaf_order)
        w.columns = ['portf_' + model + '_' + i]
        w_s = pd.concat([w_s, w], axis=1)
        

w_s['port_aver'] = w_s.mean(axis = 1 ) # add average weights 
w_s.style.format("{:.2%}").background_gradient(cmap='YlGn')

In [ ]:
# Portfolio Optimisation  - HRP 
# Train Period  
# ===========
hrp = HRPOpt(ret_under_training)
hrp.optimize()
cleaned_weights_hrp = hrp.clean_weights()
cleaned_weights_hrp = pd.DataFrame(cleaned_weights_hrp , index=[0])
cleaned_weights_hrp = pd.melt(cleaned_weights_hrp, var_name = 'fund', value_name = 'weight_hrp')
cleaned_weights_hrp

In [ ]:
cleaned_weights_hrp = w_s.reset_index().rename(columns = {'index':'fund'}).merge(cleaned_weights_hrp, left_on='fund', right_on='fund')

In [ ]:
df_weights = cleaned_weights_hrp.merge(myfunds[['fund', 'weight']], left_on='fund', right_on='fund')
#df_weights = df_weights.merge(cleaned_weights_herp, left_on='fund', right_on='fund')
df_weights

In [ ]:
#calculate portfolio returns 
def cul_port_returns(df_returns,df_weights, portfolio_name = None):
    '''
    df_returns: a wide DataFrame series with assets returns 
    df_wegiths: a long Datafrane with Fund names in the first colum 
                and weight in the other 
    '''
    weight_name = df_weights.columns[1]
    df_weights = df_weights.sort_values("fund")
    funds = list(df_weights["fund"])
    df_returns = df_returns[funds]
    df_returns =  df_returns[sorted(df_returns.columns)]
    weights = pd.Series.to_numpy(df_weights[weight_name])
    port_returns = df_returns.fillna(0).dot(weights)  # this is a workaround due to NAs 
    if portfolio_name is None: 
        port_returns = pd.Series(port_returns, name = "portfolio")
    else:
        port_returns = pd.Series(port_returns, name = portfolio_name)
    port_returns = pd.DataFrame(port_returns)
    return port_returns

In [ ]:
port_returns = []
for port in df_weights.set_index('fund').columns:
    portf_temp = cul_port_returns(df_returns,df_weights[['fund', port]], 
                                  portfolio_name= port)
    port_returns.append(portf_temp)
    
rets_portf = pd.concat(port_returns, axis = 1)

In [ ]:
df_returns = df_returns.join(rets_portf)

In [ ]:
pf.timeseries.cum_returns(df_returns[['portf_HRP_SLPM', 
                                      'weight', 
                                      'weight_hrp', 
                                      'World ex-UK']][df_returns.index>=date_to_cut], starting_value=1).plot(figsize=(12,12))
#rets_portf.plot()

In [ ]:
dct = df_returns[df_returns.index>='2021-03-01'].calc_ftca(threshold=0.8) # optimal threshold 

df_cluster = pd.DataFrame.from_dict(dct, orient='index')
df_cluster = (pd.DataFrame.from_dict(dct, orient='index').T
   .melt(var_name='cluster', value_name='fund')
   .dropna(subset=['fund']))
df_cluster['cluster'] = [f"cluster_{label}" for label in df_cluster['cluster']]
df_cluster

In [ ]:
# check some basic perfomance stats for each asset 
price_index = ffn.core.to_price_index(df_returns[df_returns.index>='2021-03-01'], start=100)
df_stats = ffn.core.GroupStats(price_index).stats
keep_index = ["total_return", "max_drawdown", "calmar", "ytd", 
              "three_month", "six_month",
              'monthly_sharpe']
df_stats[df_stats.index.isin(keep_index)].transpose().sort_values(by=['monthly_sharpe'], ascending= False).to_csv('df_stats.csv')

In [ ]:
from scipy import stats as scipy_stats
import numpy as np

def probabilistic_sharpe_ratio(returns, sr_benchmark=0.0):
    """
    Calculate the Probabilistic Sharpe Ratio (PSR).
    Parameters
    ----------
    returns: np.array, pd.Series, pd.DataFrame
        If no `returns` are passed it is mandatory to pass a `sr` and `sr_std`.
    sr_benchmark: float
        Benchmark sharpe ratio expressed in the same frequency as the other parameters.
        By default set to zero (comparing against no investment skill).
    sr: float, np.array, pd.Series, pd.DataFrame
        Sharpe ratio expressed in the same frequency as the other parameters.
    sr_std: float, np.array, pd.Series, pd.DataFrame
        Standard deviation fo the Estimated sharpe ratio,
        expressed in the same frequency as the other parameters.
    Returns
    -------
    float, pd.Series
    Notes
    -----
    PSR(SR*) = probability that SR^ > SR*
    SR^ = sharpe ratio estimated with `returns`, or `sr`
    SR* = `sr_benchmark`
    https://papers.ssrn.com/sol3/papers.cfm?abstract_id=1821643
    """
    
    ## TO DO check with annualized Returns 
    
    T = len(returns)    
    kr = pd.Series(scipy_stats.kurtosis(returns, fisher=False), index=returns.columns)
    sk = pd.Series(scipy_stats.skew(returns), index=returns.columns) # this should be set to zero for small samples 
    sr = pd.Series(returns.mean() / returns.std(ddof=1), index = returns.columns) # sharp non-annualized 
    numer = (sr - sr_benchmark)*((T-1)**0.5)
    den = (1 - sk*sr + ((kr-1)*sr**2)/4 )**0.5
    psr = scipy_stats.norm.cdf(numer/den)

    if type(returns) == pd.DataFrame:
        psr = pd.Series(psr, index=returns.columns)
    elif type(psr) not in (float, np.float64):
        psr = psr[0]

    return psr

psr = probabilistic_sharpe_ratio(df_returns, sr_benchmark=0.0)
psr = psr[psr.sort_values(ascending=False)>=0.95]


rets_benchmark = df_returns['World ex-UK']
sr_b = pd.Series(rets_benchmark.mean() / rets_benchmark.std(ddof=1)) #  sharpe 
psr2 = probabilistic_sharpe_ratio(df_returns[psr.index], sr_benchmark=sr_b[0])
psr2 = round(psr2,4).sort_values(ascending = False)
psr2

In [ ]:
#rets_portf_hrp.index = rets_portf_hrp.index.tz_localize('utc')
df_returns.index = df_returns.index.tz_localize('utc')
pf.create_returns_tear_sheet(df_returns['portfolio_HRP'],
                             benchmark_rets= df_returns['World ex-UK'],
                             live_start_date=date_to_cut)

In [ ]:
price_index = ffn.core.to_price_index(df_returns[df_returns.index>=date_to_cut], start=100)
perf = price_index[['portfolio_HRP','portfolio_herc', "portfolio_own"]].calc_stats()
perf[2].display_monthly_returns()

In [ ]:
print(perf.display())

In [ ]:
pf.timeseries.cum_returns(df_returns[['Chelverton UK','portfolio_own', 'World ex-UK']], starting_value=1).plot(figsize=(12,12))